In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
np.random.seed(42)          # cố định ngẫu nhiên -> kết quả tái lập được
print("Sẵn sàng.")

Sẵn sàng.


In [12]:
try:
    df = sns.load_dataset("titanic")
    print("Đã tải từ seaborn.")
except Exception:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    df.columns = [c.lower() for c in df.columns]
    print("Đã tải từ URL.")
df.head()

Đã tải từ seaborn.


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df['age'] = df['age'].fillna(df['age'].median())
df['fare'] = df['fare'].fillna(df['fare'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

df['sex'] = df['sex'].map({'male': 0, 'female': 1})
df = pd.get_dummies(df, columns=['embarked'], drop_first=True)

embarked_cols = [c for c in df.columns if 'embarked_' in c]
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare'] + embarked_cols

X = df[features]
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

y_pred = log_reg.predict(X_test)

print("--- ĐÁNH GIÁ MÔ HÌNH LOGISTIC REGRESSION ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

--- ĐÁNH GIÁ MÔ HÌNH LOGISTIC REGRESSION ---
Accuracy: 0.8101
Precision: 0.7857
Recall: 0.7432
F1-score: 0.7639
Confusion Matrix:
 [[90 15]
 [19 55]]


**So sánh kết quả của Logistic Regression với Linear Regression:**
- **Linear Regression:** Dự đoán các giá trị liên tục, dễ bị ảnh hưởng bởi nhiễu (outliers) và đòi hỏi tự đặt ngưỡng (ví dụ 0.5) để chia nhãn 0 hoặc 1.
- **Logistic Regression:** Sử dụng hàm Sigmoid đưa các giá trị dự đoán về khoảng xác suất từ 0 đến 1. Mô hình này được thiết kế chuyên biệt cho các bài toán phân loại.
=> **Kết luận:** Đối với bài toán phân loại nhị phân (hành khách sống/chết), Logistic Regression phù hợp hơn hẳn, thể hiện qua các chỉ số F1-score và Confusion Matrix phản ánh đúng bản chất xác suất của bài toán.

# Data Processing – Seeds Dataset

Notebook này xử lý `seeds_dataset.csv` và tạo hai file:

- `seeds_train.csv`
- `seeds_test.csv`

Quy trình:

1. Đọc dữ liệu.
2. Chuẩn hóa tên cột.
3. Kiểm tra kiểu dữ liệu, missing values và duplicate.
4. Kiểm tra phân bố target.
5. Chia train/test theo tỷ lệ 80/20 với `stratify`.
6. Lưu hai file CSV.

> Chưa chuẩn hóa bằng `StandardScaler` ở notebook này. Scaling sẽ được thực hiện khi train mô hình để tránh data leakage.


In [14]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


## 1. Khai báo đường dẫn

In [15]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("Dry_Bean_Dataset") / "Dry_Bean_Dataset.xlsx"

OUTPUT_DIR = DATA_PATH.parent
TRAIN_OUTPUT_PATH = OUTPUT_DIR / "dry_bean_train.csv"
TEST_OUTPUT_PATH = OUTPUT_DIR / "dry_bean_test.csv"

print("Đường dẫn:", DATA_PATH.resolve())
print("File tồn tại:", DATA_PATH.exists())

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file tại: {DATA_PATH.resolve()}"
    )

Đường dẫn: /content/Dry_Bean_Dataset/Dry_Bean_Dataset.xlsx
File tồn tại: True


## 2. Đọc dữ liệu

In [16]:
%pip install openpyxl

In [ ]:
df = pd.read_excel(
    DATA_PATH,
    engine="openpyxl"
)

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r"[^a-z0-9]+", "_", regex=True)
      .str.strip("_")
)

print(df.shape)
print(df["class"].head())
print(df["class"].unique())

## 3. Chuẩn hóa tên cột

In [ ]:
target = "class"

numeric_columns = [
    column for column in df.columns
    if column != target
]

df[numeric_columns] = df[numeric_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

# Làm sạch target dạng chữ
df[target] = (
    df[target]
    .astype(str)
    .str.strip()
    .str.upper()
)

## 4. Kiểm tra cấu trúc dữ liệu

In [ ]:
print("Thông tin dữ liệu:")
df.info()

print("\nThống kê mô tả:")
display(df.describe(include="all").T)


## 5. Kiểm tra missing values

In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)

display(missing)
print("Tổng số missing values:", df.isna().sum().sum())


In [ ]:
rows_before_missing = len(df)

df = df.dropna().reset_index(drop=True)

rows_after_missing = len(df)

print("Số hàng trước khi bỏ missing:", rows_before_missing)
print("Số hàng sau khi bỏ missing:", rows_after_missing)
print("Số hàng đã bỏ:", rows_before_missing - rows_after_missing)


## 6. Kiểm tra và xóa duplicate

In [ ]:
duplicate_count = df.duplicated().sum()

print("Số dòng trùng hoàn toàn:", duplicate_count)

df = df.drop_duplicates().reset_index(drop=True)

print("Kích thước sau khi xóa duplicate:", df.shape)


## 7. Xác định feature và target

In [ ]:
target = "class"

# Tất cả các cột ngoại trừ target đều là feature
features = [
    column for column in df.columns
    if column != target
]

X = df[features].copy()
y = df[target].copy()

print("Danh sách feature:")
print(features)

print("\nSố feature:", len(features))
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nPhân bố target:")
print(y.value_counts())

## 9. Kiểm tra phân bố target

In [ ]:
print("Số lượng từng lớp:")
print(y.value_counts().sort_index())

print("\nTỷ lệ từng lớp:")
print(y.value_counts(normalize=True).sort_index().round(4))

print("\nCác nhãn có trong target:")
print(sorted(y.unique()))


## 10. Chia train/test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


## 11. Ghép feature và target

In [ ]:
train_df = X_train.copy()
train_df[target] = y_train

test_df = X_test.copy()
test_df[target] = y_test

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

display(train_df.head())


## 12. Kiểm tra phân bố lớp sau khi chia

In [ ]:
print("Phân bố lớp trong train:")
print(train_df[target].value_counts().sort_index())

print("\nTỷ lệ lớp trong train:")
print(train_df[target].value_counts(normalize=True).sort_index().round(4))

print("\nPhân bố lớp trong test:")
print(test_df[target].value_counts().sort_index())

print("\nTỷ lệ lớp trong test:")
print(test_df[target].value_counts(normalize=True).sort_index().round(4))


## 13. Lưu thành hai file CSV

In [ ]:
train_df.to_csv(TRAIN_OUTPUT_PATH, index=False)
test_df.to_csv(TEST_OUTPUT_PATH, index=False)

print("Đã lưu file train:", TRAIN_OUTPUT_PATH.resolve())
print("Đã lưu file test:", TEST_OUTPUT_PATH.resolve())


## 14. Kiểm tra lại file đã lưu

In [ ]:
saved_train = pd.read_csv(TRAIN_OUTPUT_PATH)
saved_test = pd.read_csv(TEST_OUTPUT_PATH)

print("Saved train shape:", saved_train.shape)
print("Saved test shape:", saved_test.shape)

print("\nMissing trong train:", saved_train.isna().sum().sum())
print("Missing trong test:", saved_test.isna().sum().sum())

print("\nCác cột trong train:")
print(saved_train.columns.tolist())


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

# 1. Tách features và target từ dữ liệu đã lưu ở cell trên
X_train_b = saved_train.drop(columns=["class"])
y_train_b = saved_train["class"]

X_test_b = saved_test.drop(columns=["class"])
y_test_b = saved_test["class"]

# 2. Chuẩn hóa dữ liệu (Scaling) để tránh data leakage
scaler = StandardScaler()
X_train_b_scaled = scaler.fit_transform(X_train_b)
X_test_b_scaled = scaler.transform(X_test_b)

# 3. Thuật toán 1: Logistic Regression
lr_model = LogisticRegression(max_iter=2000, multi_class='multinomial')
lr_model.fit(X_train_b_scaled, y_train_b)
y_pred_lr = lr_model.predict(X_test_b_scaled)

print("=== KẾT QUẢ LOGISTIC REGRESSION ===")
print(classification_report(y_test_b, y_pred_lr))

# 4. Thuật toán 2: K-Nearest Neighbors (KNN)
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_b_scaled, y_train_b)
y_pred_knn = knn_model.predict(X_test_b_scaled)

print("\n=== KẾT QUẢ K-NEAREST NEIGHBORS (KNN) ===")
print(classification_report(y_test_b, y_pred_knn))